### **Memuat Dataset**

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv("DataFiles/data_cleaning.csv")

# Ringkasan dimensi dataset
df_summary = pd.DataFrame({
    "Metrik": ["Total Baris", "Total Kolom"],
    "Nilai": [df.shape[0], df.shape[1]]
})

# Audit struktur kolom
dup_cols = df.columns[df.columns.duplicated()].tolist()
whitespace_cols = [c for c in df.columns if c != c.strip()]

df_audit = pd.DataFrame({
    "Jenis Pengecekan": ["Status Duplikasi", "Status Spasi Tersembunyi"],
    "Status": [
        "✅ Aman" if not dup_cols else "⚠️ Terdeteksi",
        "✅ Aman" if not whitespace_cols else "⚠️ Terdeteksi"
    ],
    "Detail Kolom": [
        ", ".join(dup_cols) if dup_cols else "-",
        ", ".join(whitespace_cols) if whitespace_cols else "-"
    ]
})

print("📊 RINGKASAN DATASET")
display(df_summary)

print("\n🔍 AUDIT STRUKTUR KOLOM")
display(df_audit)

print("\n👀 SAMPEL 2 DATA TERATAS")
display(df.head(2))

📊 RINGKASAN DATASET


,Metrik,Nilai
0,Total Baris,11429
1,Total Kolom,83



🔍 AUDIT STRUKTUR KOLOM


,Jenis Pengecekan,Status,Detail Kolom
0,Status Duplikasi,✅ Aman,-
1,Status Spasi Tersembunyi,✅ Aman,-



👀 SAMPEL 2 DATA TERATAS


,url,length_url,length_hostname,ip,nb_dots,nb_hyphens,nb_at,nb_qm,nb_and,nb_eq,...,domain_in_title,domain_with_copyright,whois_registered_domain,domain_registration_length,domain_age,web_traffic,dns_record,google_index,page_rank,label
0,http://www.crestonwood.com/router.php,37,19,0,3,0,0,0,0,0,...,0,1,0,45,-1,0,1,1,4,0
1,http://shadetreetechnology.com/V4/validation/a...,77,23,1,1,0,0,0,0,0,...,1,0,0,77,5767,0,0,1,2,1


## **Split Data**

In [2]:
from sklearn.model_selection import train_test_split

# Persiapan kolom
df.columns = df.columns.str.strip()

TARGET_COL = "label"
ACTIVE_TRAIN_FEATURES = "hybrid81"  # "url37" atau "hybrid81"

url_features_37 = [
    "length_url", "length_hostname", "ip", "nb_dots", "nb_hyphens", "nb_at", "nb_qm", "nb_and",
    "nb_eq", "nb_underscore", "nb_tilde", "nb_percent", "nb_slash", "nb_star", "nb_colon",
    "nb_comma", "nb_semicolumn", "nb_dollar", "nb_space", "nb_www", "nb_com", "nb_dslash",
    "http_in_path", "https_token", "ratio_digits_url", "ratio_digits_host", "punycode", "port",
    "tld_in_path", "tld_in_subdomain", "abnormal_subdomain", "nb_subdomains", "prefix_suffix",
    "random_domain", "shortening_service", "path_extension", "nb_redirection"
]

id_cols = [c for c in ["url"] if c in df.columns]
all_features = [c for c in df.columns if c not in [TARGET_COL] + id_cols]

webcontent_features_44 = [c for c in all_features if c not in url_features_37]
hybrid_features_81 = url_features_37 + webcontent_features_44
feature_candidates = hybrid_features_81 if ACTIVE_TRAIN_FEATURES == "hybrid81" else url_features_37

# Cegah error jika ada fitur kandidat yang tidak tersedia
available_features = [c for c in feature_candidates if c in df.columns]
missing_features = [c for c in feature_candidates if c not in df.columns]
if missing_features:
    print(f"⚠️ Fitur tidak ditemukan dan diabaikan: {len(missing_features)} kolom")

# Siapkan X dan y
X_all = df[available_features].apply(pd.to_numeric, errors="coerce")
y_all = pd.to_numeric(df[TARGET_COL], errors="coerce").fillna(0).astype("int64")

# Split 80:20
X_train, X_test, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=0.2,
    stratify=y_all,
    random_state=12
)

# Filter fitur valid (minimal ada nilai non-NaN di train)
selected_features = X_train.columns[X_train.notna().any()].tolist()
X_train = X_train[selected_features]
X_test = X_test[selected_features]

# Imputasi median dari train
med_all = X_train.median(numeric_only=True)
X_train = X_train.fillna(med_all)
X_test = X_test.fillna(med_all)

# Output ringkasan
df_config = pd.DataFrame({
    "Pengaturan Pemodelan": [
        "Kolom Target", "Mode Pelatihan",
        "Total Kandidat Fitur", "Fitur Tersedia",
        "Jumlah Fitur Valid", "Strategi Imputasi"
    ],
    "Keterangan": [
        TARGET_COL,
        ACTIVE_TRAIN_FEATURES,
        len(feature_candidates),
        len(available_features),
        len(selected_features),
        "Median (dari Data Train)"
    ]
})

df_split = pd.DataFrame({
    "Dataset": ["Train Data (80%)", "Test Data (20%)"],
    "Jumlah Baris": [X_train.shape[0], X_test.shape[0]],
    "Jumlah Kolom (Fitur)": [X_train.shape[1], X_test.shape[1]]
})

print("⚙️ KONFIGURASI:")
display(df_config)

print("\n📊 DIMENSI DATASET:")
display(df_split)

⚙️ KONFIGURASI:


,Pengaturan Pemodelan,Keterangan
0,Kolom Target,label
1,Mode Pelatihan,hybrid81
2,Total Kandidat Fitur,81
3,Fitur Tersedia,81
4,Jumlah Fitur Valid,81
5,Strategi Imputasi,Median (dari Data Train)



📊 DIMENSI DATASET:


,Dataset,Jumlah Baris,Jumlah Kolom (Fitur)
0,Train Data (80%),9143,81
1,Test Data (20%),2286,81


### **Rule-Based Filtering (Pre-filter)**

In [3]:
import re
import math
import ipaddress
from urllib.parse import urlparse
import tldextract
from sklearn.metrics import confusion_matrix

SHORTENERS = {
    "bit.ly", "goo.gl", "tinyurl.com", "ow.ly", "t.co", "is.gd", "buff.ly",
    "adf.ly", "bit.do", "cutt.ly"
}
SUSPICIOUS_TLD = [
    "zip", "xyz", "top", "tk", "ga", "ml", "gq", "cf", "pw", "cc", "club",
    "ws", "biz", "online", "site", "live", "work", "icu", "info",
    "cn", "ru", "loan", "download", "click"
]
STANDARD_PORTS = {21, 22, 23, 80, 443, 445, 1433, 1521, 3306, 3389}
PHISH_HINTS = [
    "login", "verify", "update", "secure", "account", "bank",
    "paypal", "apple", "microsoft", "confirm", "signin", "password"
]
BRANDS = [
    "google", "facebook", "apple", "microsoft", "amazon", "paypal",
    "instagram", "whatsapp", "telegram", "netflix", "github", "linkedin"
]
PREFILTER_HARD_PHISHING_SCORE = 7

def entropy(s):
    if not s:
        return 0.0
    probs = [s.count(c) / len(s) for c in set(s)]
    return -sum(p * math.log2(p) for p in probs)

def parse_url(url):
    u = (url or "").strip()
    if not u.startswith(("http://", "https://")):
        u = "http://" + u
    parsed = urlparse(u)
    hostname = (parsed.hostname or "").lower()
    path = parsed.path or ""
    return u, parsed, hostname, path

def is_ip(hostname):
    try:
        ipaddress.ip_address(hostname)
        return 1
    except Exception:
        return 0

def extract_url_features(url):
    full, parsed, hostname, path = parse_url(url)
    ext = tldextract.extract(full)
    subdomain = (ext.subdomain or "").lower()
    domain = (ext.domain or "").lower()
    suffix = (ext.suffix or "").lower()

    digits_url = sum(c.isdigit() for c in full)
    digits_host = sum(c.isdigit() for c in hostname)

    random_domain = 1 if entropy(domain) > 3.5 else 0
    shortening_service = 1 if any(s in hostname for s in SHORTENERS) else 0
    prefix_suffix = 1 if "-" in domain else 0
    path_extension = 1 if "." in path.split("/")[-1] else 0
    nb_redirection = max(full.lower().count("http") - 1, 0)

    try:
        parsed_port = parsed.port
    except ValueError:
        parsed_port = None
    port_flag = 1 if (parsed_port is not None and parsed_port not in STANDARD_PORTS) else 0

    tld_last_label = suffix.split(".")[-1] if suffix else ""
    suspicious_tld_flag = 1 if (suffix in SUSPICIOUS_TLD or tld_last_label in SUSPICIOUS_TLD) else 0

    domain_in_brand = 1 if any(b in domain for b in BRANDS) else 0
    brand_in_subdomain = 1 if any(b in subdomain for b in BRANDS) else 0
    brand_in_path = 1 if any(b in path.lower() for b in BRANDS) else 0

    statistical_report = 1 if (
        suspicious_tld_flag == 1
        or is_ip(hostname) == 1
        or full.count("@") >= 1
        or random_domain == 1
    ) else 0

    return {
        "length_url": len(full),
        "length_hostname": len(hostname),
        "ip": is_ip(hostname),
        "nb_dots": full.count("."),
        "nb_hyphens": full.count("-"),
        "nb_at": full.count("@"),
        "nb_qm": full.count("?"),
        "nb_and": full.count("&"),
        "nb_eq": full.count("="),
        "nb_underscore": full.count("_"),
        "nb_tilde": full.count("~"),
        "nb_percent": full.count("%"),
        "nb_slash": full.count("/"),
        "nb_star": full.count("*"),
        "nb_colon": full.count(":"),
        "nb_comma": full.count(","),
        "nb_semicolumn": full.count(";"),
        "nb_dollar": full.count("$"),
        "nb_space": full.count(" "),
        "nb_www": 1 if "www" in hostname else 0,
        "nb_com": full.count(".com"),
        "nb_dslash": full.count("//"),
        "http_in_path": 1 if "http" in path else 0,
        "https_token": 1 if "https" in full.replace("https://", "") else 0,
        "ratio_digits_url": digits_url / max(len(full), 1),
        "ratio_digits_host": digits_host / max(len(hostname), 1),
        "punycode": 1 if "xn--" in hostname else 0,
        "port": port_flag,
        "tld_in_path": 1 if suffix and (suffix in path) else 0,
        "tld_in_subdomain": 1 if suffix and (suffix in subdomain) else 0,
        "abnormal_subdomain": 1 if ("http" in subdomain or "https" in subdomain) else 0,
        "nb_subdomains": len(subdomain.split(".")) if subdomain else 0,
        "prefix_suffix": prefix_suffix,
        "random_domain": random_domain,
        "shortening_service": shortening_service,
        "path_extension": path_extension,
        "nb_redirection": nb_redirection,
        "nb_external_redirection": 0,
        "length_words_raw": len(re.findall(r"[A-Za-z0-9]+", full.lower())),
        "char_repeat": sum(1 for i in range(1, len(full)) if full[i] == full[i - 1]),
        "shortest_words_raw": 0,
        "shortest_word_host": 0,
        "shortest_word_path": 0,
        "longest_words_raw": 0,
        "longest_word_host": 0,
        "longest_word_path": 0,
        "avg_words_raw": 0.0,
        "avg_word_host": 0.0,
        "avg_word_path": 0.0,
        "phish_hints": sum(1 for k in PHISH_HINTS if k in full.lower()),
        "domain_in_brand": domain_in_brand,
        "brand_in_subdomain": brand_in_subdomain,
        "brand_in_path": brand_in_path,
        "suspicious_tld": suspicious_tld_flag,
        "statistical_report": statistical_report,
    }

def rule_based_eval(feats):
    very_important = {
        "suspicious_tld": (feats.get("suspicious_tld", 0) == 1),
        "nb_at": (feats.get("nb_at", 0) >= 1),
        "ip": (feats.get("ip", 0) == 1),
        "nb_underscore": (feats.get("nb_underscore", 0) > 3),
    }
    important = {
        "ratio_digits_url": (feats.get("ratio_digits_url", 0) > 0.3),
        "nb_subdomains": (feats.get("nb_subdomains", 0) > 3),
        "nb_percent": (feats.get("nb_percent", 0) > 5),
        "nb_tilde": (feats.get("nb_tilde", 0) >= 1),
        "nb_semicolumn": (feats.get("nb_semicolumn", 0) >= 1),
        "nb_star": (feats.get("nb_star", 0) >= 1),
        "nb_comma": (feats.get("nb_comma", 0) >= 1),
        "random_domain": (feats.get("random_domain", 0) == 1),
    }
    less_important = {
        "length_hostname": (feats.get("length_hostname", 0) > 30),
        "nb_dollar": (feats.get("nb_dollar", 0) >= 1),
        "nb_qm": (feats.get("nb_qm", 0) > 2),
        "nb_colon": (feats.get("nb_colon", 0) > 1),
        "nb_eq": (feats.get("nb_eq", 0) > 8),
        "nb_dots": (feats.get("nb_dots", 0) > 4),
        "nb_slash": (feats.get("nb_slash", 0) > 7),
        "nb_and": (feats.get("nb_and", 0) > 3),
        "nb_hyphens": (feats.get("nb_hyphens", 0) > 3),
        "http_in_path": (feats.get("http_in_path", 0) == 1),
        "https_token": (feats.get("https_token", 0) == 1),
        "port": (feats.get("port", 0) == 1),
        "shortening_service": (feats.get("shortening_service", 0) == 1),
    }

    vi_count = sum(very_important.values())
    imp_count = sum(important.values())
    less_count = sum(less_important.values())

    risk_score = (3 * vi_count) + (2 * imp_count) + less_count
    rule_flag = int((vi_count >= 1) or (risk_score >= PREFILTER_HARD_PHISHING_SCORE))
    category = "Phishing" if rule_flag == 1 else "Suspicious"

    return risk_score, category, rule_flag, {
        "vi_count": vi_count,
        "imp_count": imp_count,
        "less_count": less_count,
    }

def predict_url(url):
    feats = extract_url_features(url)
    return rule_based_eval(feats)

# Evaluasi rule-based pada data yang sudah ada
if "label" in df.columns:
    y_true = pd.to_numeric(df["label"], errors="coerce").fillna(0).astype(int)

    X_rules = df.drop(columns=["label"], errors="ignore")
    def _col(c):
        return X_rules[c] if c in X_rules.columns else pd.Series(0, index=X_rules.index)

    vi_count = (
        (_col("suspicious_tld") == 1).astype(int) +
        (_col("nb_at") >= 1).astype(int) +
        (_col("ip") == 1).astype(int) +
        (_col("nb_underscore") > 3).astype(int)
    )

    imp_count = (
        (_col("ratio_digits_url") > 0.3).astype(int) +
        (_col("nb_subdomains") > 3).astype(int) +
        (_col("nb_percent") > 5).astype(int) +
        (_col("nb_tilde") >= 1).astype(int) +
        (_col("nb_semicolumn") >= 1).astype(int) +
        (_col("nb_star") >= 1).astype(int) +
        (_col("nb_comma") >= 1).astype(int) +
        (_col("random_domain") == 1).astype(int)
    )

    less_count = (
        (_col("length_hostname") > 30).astype(int) +
        (_col("nb_dollar") >= 1).astype(int) +
        (_col("nb_qm") > 2).astype(int) +
        (_col("nb_colon") > 1).astype(int) +
        (_col("nb_eq") > 8).astype(int) +
        (_col("nb_dots") > 4).astype(int) +
        (_col("nb_slash") > 7).astype(int) +
        (_col("nb_and") > 3).astype(int) +
        (_col("nb_hyphens") > 3).astype(int) +
        (_col("http_in_path") == 1).astype(int) +
        (_col("https_token") == 1).astype(int) +
        (_col("port") == 1).astype(int) +
        (_col("shortening_service") == 1).astype(int)
    )

    risk_score = (3 * vi_count) + (2 * imp_count) + less_count
    y_pred_rule = ((vi_count >= 1) | (risk_score >= PREFILTER_HARD_PHISHING_SCORE)).astype(int)

    cm = confusion_matrix(y_true, y_pred_rule, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    cm_display = pd.DataFrame(
        cm,
        index=["Actual Benign (0)", "Actual Phishing (1)"],
        columns=["Predicted Benign (0)", "Predicted Phishing (1)"]
    )

    print("🎯 HASIL EVALUASI RULE-BASED")
    display(cm_display)

🎯 HASIL EVALUASI RULE-BASED


,Predicted Benign (0),Predicted Phishing (1)
Actual Benign (0),5390,325
Actual Phishing (1),3826,1888


## **Train Model**

In [4]:
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import time

ga_rf_params = {
    "n_estimators": 428,
    "max_depth": 16,
    "min_samples_split": 4,
    "min_samples_leaf": 1,
    "max_features": "log2",
    "random_state": 12,
    "n_jobs": -1
}

ga_xgb_params = {
    "n_estimators": 356,
    "learning_rate": 0.24470280263728614,
    "max_depth": 5,
    "subsample": 0.799109701671054,
    "colsample_bytree": 0.7933724339946145,
    "min_child_weight": 6.656725584883891,
    "gamma": 1.295747350495099,
    "reg_alpha": 0.8880322724647041,
    "reg_lambda": 2.2701556904961406,
    "eval_metric": "logloss",
    "random_state": 12,
    "n_jobs": -1
}

rf_ga = RandomForestClassifier(**ga_rf_params)
xgb_ga = XGBClassifier(**ga_xgb_params)

print("⏳ Training Random Forest (GA tuning)...")
t0 = time.time()
rf_ga.fit(X_train, y_train)
print(f"Done in {time.time()-t0:.2f}s")

print("⏳ Training XGBoost (GA tuning)...")
t0 = time.time()
xgb_ga.fit(X_train, y_train)
print(f"Done in {time.time()-t0:.2f}s")

base_learners_ga = [
    ("rf", rf_ga),
    ("xgb", xgb_ga),
]
meta_learner_ga = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=12)

stack_ga = StackingClassifier(
    estimators=base_learners_ga,
    final_estimator=meta_learner_ga,
    stack_method="predict_proba",
    n_jobs=-1
)

print("⏳ Training Stacking (GA tuning)...")
t0 = time.time()
stack_ga.fit(X_train, y_train)
print(f"Done in {time.time()-t0:.2f}s")

⏳ Training Random Forest (GA tuning)...
Done in 0.87s
⏳ Training XGBoost (GA tuning)...
Done in 1.67s
⏳ Training Stacking (GA tuning)...
Done in 11.42s


## **Confution Matrix**

In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

def evaluate_model(name, model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    y_prob = model.predict_proba(X_eval)[:, 1] if hasattr(model, "predict_proba") else y_pred

    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    try:
        auc = roc_auc_score(y_eval, y_prob)
    except Exception:
        auc = np.nan

    cm_table = pd.DataFrame(
        confusion_matrix(y_eval, y_pred, labels=[0, 1]),
        index=["Actual Benign (0)", "Actual Phishing (1)"],
        columns=["Predicted Benign (0)", "Predicted Phishing (1)"]
    )

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_eval, y_pred),
        "Precision": precision_score(y_eval, y_pred, zero_division=0),
        "Recall": recall_score(y_eval, y_pred, zero_division=0),
        "F1 Score": f1_score(y_eval, y_pred, zero_division=0),
        "AUC": auc,
        "✅ TP": int(tp),
        "✅ TN": int(tn),
        "❌ FP": int(fp),
        "❌ FN": int(fn),
        "cm_table": cm_table
    }

evaluations = [
    evaluate_model("Random Forest (GA)", rf_ga, X_test, y_test),
    evaluate_model("XGBoost (GA)", xgb_ga, X_test, y_test),
    evaluate_model("Stacking (RF+XGB+LR, GA)", stack_ga, X_test, y_test),
]

# Leaderboard metrik
detail_df = pd.DataFrame([{k: v for k, v in e.items() if k != "cm_table"} for e in evaluations])

print("📊 PERBANDINGAN PERFORMA MODEL PADA DATA TEST")
display(
    detail_df.style
    .format({
        "Accuracy": "{:.3f}",
        "Precision": "{:.3f}",
        "Recall": "{:.3f}",
        "F1 Score": "{:.3f}",
        "AUC": "{:.3f}"
    })
    .background_gradient(cmap="Greens", subset=["Accuracy", "Precision", "Recall", "F1 Score", "AUC"])
    .background_gradient(cmap="Blues", subset=["✅ TP", "✅ TN"])
    .background_gradient(cmap="Reds", subset=["❌ FP", "❌ FN"])
    .set_properties(**{"text-align": "center", "vertical-align": "middle"})
)

# Confusion matrix per model
for e in evaluations:
    print(f"\n🧩 CONFUSION MATRIX - {e['Model']}")
    display(e["cm_table"])

📊 PERBANDINGAN PERFORMA MODEL PADA DATA TEST


,Model,Accuracy,Precision,Recall,F1 Score,AUC,✅ TP,✅ TN,❌ FP,❌ FN
0,Random Forest (GA),0.972,0.972,0.972,0.972,0.995,1111,1111,32,32
1,XGBoost (GA),0.974,0.974,0.974,0.974,0.996,1113,1113,30,30
2,"Stacking (RF+XGB+LR, GA)",0.973,0.973,0.974,0.973,0.996,1113,1112,31,30



🧩 CONFUSION MATRIX - Random Forest (GA)


,Predicted Benign (0),Predicted Phishing (1)
Actual Benign (0),1111,32
Actual Phishing (1),32,1111



🧩 CONFUSION MATRIX - XGBoost (GA)


,Predicted Benign (0),Predicted Phishing (1)
Actual Benign (0),1113,30
Actual Phishing (1),30,1113



🧩 CONFUSION MATRIX - Stacking (RF+XGB+LR, GA)


,Predicted Benign (0),Predicted Phishing (1)
Actual Benign (0),1112,31
Actual Phishing (1),30,1113


## **Implementasi Pipeline Rule-Based & Model ML (GA Tuning) untuk Evaluasi Matrix**

Pada bagian ini, pipeline prediksi dan evaluasi akan disusun ulang agar sesuai dengan logika di app.py, termasuk:


- Fungsi utility untuk feature extraction dan rule-based
- Pipeline prediksi model (Random Forest, XGBoost, Stacking/Meta) dengan parameter hasil tuning GA
- Evaluasi confusion matrix dan metrik utama (akurasi, presisi, recall, F1, AUC)



> **Catatan:**
> - Pastikan model sudah dilatih dengan parameter hasil tuning GA.
> - Jika ingin menguji URL baru, gunakan fungsi prediksi yang disediakan.
